# Nemotron v9 — Final, no compromises

Goal: **score > 0.85**, 3 epochs, 0% data loss, runtime ≤ 11 hrs, max LoRA performance.

**Stack** (all proven on the 0.85 LB notebook — only the data + alpha + epochs differ):

| component | value |
|---|---|
| framework | Unsloth `FastLanguageModel` (no `trust_remote_code` import hell) |
| Mamba CUDA fast path | ENABLED (mayukh18 wheels, pre-built for sm_120) |
| loss | Cut Cross-Entropy (no `[B,T,131072]` logits) |
| LoRA target | `q,k,v,o, up,down, in_proj,out_proj, lm_head` |
| LoRA rank / alpha | 32 / **64** (vs 0.85's 32/32 — sharper updates) |
| MoE tied LoRA | ON (Tinker convention, sum-grad sync) |
| precision | fp32 LoRA + bf16 base + fp32 MoE router (Nemotron-H requirement) |
| sequence | tail-truncate to 8192 → 100% data kept, answers preserved |
| batch | 32 effective (micro=4) |
| epochs | **3** |
| optimizer | `torch.optim.AdamW` β=(0.9,0.95), wd=0, eps=1e-8 |
| LR schedule | linear decay 2e-4 → 0 |

**Required Kaggle inputs:**
- `mayukh18/nemotron-packages` — Unsloth + Blackwell mamba/causal_conv1d wheels
- `metric/nemotron-3-nano-30b-a3b-bf16` (model)
- Your `nemotron-categorical-splits` dataset (or any path in `DATA_DIR_CANDIDATES`)

**Internet:** can be OFF — all installs are from offline kaggle datasets.


In [1]:
# ── Cell 1: Config ───────────────────────────────────────────────────
LORA_RANK    = 32       # eval-server cap
LORA_ALPHA   = 64       # 2x vs 0.85 — sharper LoRA updates
LORA_DROPOUT = 0.0      # eval-server contract

MAX_SEQ_LEN       = 8192     # eval cap; tail-truncate longer samples
MICRO_BATCH_SIZE  = 4
BATCH_SIZE        = 32       # effective via grad-accum (32/4 = 8 micro-steps)
LEARNING_RATE     = 2e-4
NUM_EPOCHS        = 3        # 3x vs 0.85's 1

MOE_TIE_WEIGHTS   = True

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "up_proj", "down_proj",
    "in_proj", "out_proj",
    "lm_head",
]

CATEGORY_FILES = [
    "train_cot_bit_manipulation.jsonl",
    "train_cot_cipher.jsonl",
    "train_cot_cryptarithm_deduce.jsonl",
    "train_cot_cryptarithm_guess.jsonl",
    "train_cot_equation_numeric_deduce.jsonl",
    "train_cot_equation_numeric_guess.jsonl",
    "train_cot_gravity.jsonl",
    "train_cot_numeral.jsonl",
    "train_cot_unit_conversion.jsonl",
]

DATA_DIR_CANDIDATES = [
    "/kaggle/input/datasets/asharamkanderiwal/nvidia-dataset/all_categorical_splits",

]


In [2]:
# ── Cell 2: Env detect ───────────────────────────────────────────────
import os
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
print(f"IS_KAGGLE={IS_KAGGLE}")


IS_KAGGLE=True


In [3]:
# ── Cell 3: Offline installs (Kaggle only) ───────────────────────────
# Critical rules learned the hard way:
#   1) --target /kaggle/working/packages so we never write to the read-only
#      utility-script dir.
#   2) --no-deps so pip doesn't try to upgrade Kaggle's bundled torch/cuda.
#   3) Do NOT shadow PyO3 native modules (safetensors, tokenizers). They
#      can only init once per Python process. Use Kaggle's bundled ones.
#      We also AGGRESSIVELY DELETE any leftover copies from previous runs.
#   4) Pin huggingface_hub<1.0 in our TARGET_DIR; Kaggle ships hf_hub 1.8.0
#      system-wide and our older transformers asserts <1.0.
if IS_KAGGLE:
    import subprocess, sys, shutil
    from pathlib import Path

    TARGET_DIR = "/kaggle/working/packages"
    PKGS_DIR   = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    WHEELS_DIR = "/kaggle/input/datasets/mayukh18/nemotron-packages"
    os.makedirs(TARGET_DIR, exist_ok=True)

    # ── Step 0: PURGE leftover PyO3-native packages from prior runs ──
    # If a previous run installed safetensors/tokenizers into TARGET_DIR,
    # they will shadow Kaggle's already-loaded native .so files and crash
    # with "PyO3 modules ... may only be initialized once". Wipe them now,
    # BEFORE any import attempts use TARGET_DIR.
    _PYO3_DIRS = ["safetensors", "tokenizers", "_safetensors_rust",
                  "safetensors-*.dist-info", "tokenizers-*.dist-info"]
    import glob as _glob
    for pat in _PYO3_DIRS:
        for path in _glob.glob(os.path.join(TARGET_DIR, pat)):
            try:
                if os.path.isdir(path):
                    shutil.rmtree(path)
                else:
                    os.remove(path)
                print(f"  [purge] {path}")
            except Exception as e:
                print(f"  [purge fail] {path}: {e}")

    # Also drop any pre-imported safetensors/tokenizers that may have come
    # from TARGET_DIR in this same session (re-running cells). They should
    # come from the SYSTEM path going forward.
    for _m in list(sys.modules):
        if _m.split(".")[0] in ("safetensors", "tokenizers"):
            _f = getattr(sys.modules[_m], "__file__", "") or ""
            if TARGET_DIR in _f:
                del sys.modules[_m]

    if TARGET_DIR not in sys.path:
        sys.path.insert(0, TARGET_DIR)  # FRONT — our pure-Python pkgs win

    # ── Step 1: pure-Python deps (NO safetensors / tokenizers) ──────
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index", "--find-links", PKGS_DIR,
        "--target", TARGET_DIR, "--no-deps",
        "unsloth", "unsloth_zoo", "trl", "peft", "transformers",
        "datasets", "accelerate", "bitsandbytes", "cut-cross-entropy",
        "huggingface_hub<1.0",
    ])

    # ── Step 2: Blackwell mamba CUDA wheels (pre-built for sm_120) ──
    for whl_name in [
        "causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
        "mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
    ]:
        whl = os.path.join(WHEELS_DIR, whl_name)
        if os.path.exists(whl):
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "--target", TARGET_DIR, "--no-deps", whl,
            ])
        else:
            print(f"[warn] missing wheel: {whl}")

    # ── Step 3: PURGE AGAIN. The pip install above pulls transitive
    # ── safetensors/tokenizers wheels for some packages even with
    # ── --no-deps if they're listed as direct deps. Clean them out a
    # ── second time, immediately after install.
    for pat in _PYO3_DIRS:
        for path in _glob.glob(os.path.join(TARGET_DIR, pat)):
            try:
                if os.path.isdir(path):
                    shutil.rmtree(path)
                else:
                    os.remove(path)
                print(f"  [post-purge] {path}")
            except Exception as e:
                print(f"  [post-purge fail] {path}: {e}")

    # Resolve any .pth files (namespace pkgs) our installs dropped
    for pth in Path(TARGET_DIR).glob("*.pth"):
        try:
            with pth.open() as fp:
                rel = fp.read().strip()
                p = pth.parent / rel
                if p.exists() and str(p) not in sys.path:
                    sys.path.append(str(p))
        except Exception:
            pass

    # Purge Kaggle utility-script paths that ship a different mamba_ssm
    _bad = ("nvidia_utility_script", "nvidia-utility-script")
    sys.path[:] = [p for p in sys.path if not any(b in p for b in _bad)]
    for _m in list(sys.modules):
        _f = getattr(sys.modules[_m], "__file__", "") or ""
        if any(b in _f for b in _bad):
            del sys.modules[_m]

    # Drop pre-imported PURE-PYTHON modules so the next import picks up
    # our pinned versions from TARGET_DIR. NOT safetensors/tokenizers.
    _PURE = (
        "huggingface_hub", "transformers", "datasets", "trl", "peft",
        "unsloth", "unsloth_zoo", "accelerate",
    )
    for _m in list(sys.modules):
        if _m.split(".")[0] in _PURE:
            del sys.modules[_m]

    # ── Sanity check ────────────────────────────────────────────────
    import huggingface_hub, transformers, peft, trl, safetensors, tokenizers
    print(f"  huggingface_hub : {huggingface_hub.__version__}  ({huggingface_hub.__file__})")
    print(f"  transformers    : {transformers.__version__}  ({transformers.__file__})")
    print(f"  peft            : {peft.__version__}")
    print(f"  trl             : {trl.__version__}")
    print(f"  safetensors     : {safetensors.__version__}  ({safetensors.__file__})")
    print(f"  tokenizers      : {tokenizers.__version__}  ({tokenizers.__file__})")

    assert TARGET_DIR not in safetensors.__file__, \
        f"safetensors loaded from TARGET_DIR ({safetensors.__file__}) — purge failed"
    assert TARGET_DIR not in tokenizers.__file__, \
        f"tokenizers loaded from TARGET_DIR ({tokenizers.__file__}) — purge failed"

    print("[ok] installs complete -> /kaggle/working/packages")


  [purge] /kaggle/working/packages/safetensors
  [purge] /kaggle/working/packages/tokenizers
  [purge] /kaggle/working/packages/safetensors-0.7.0.dist-info
  [purge] /kaggle/working/packages/tokenizers-0.22.2.dist-info


ImportError: huggingface-hub>=0.34.0,<1.0 is required for a normal functioning of this module, but found huggingface-hub==1.8.0.
Try: `pip install transformers -U` or `pip install -e '.[dev]'` if you're working with git main

In [ ]:
# ── Cell 4: Full training pipeline ───────────────────────────────────
def run_training() -> None:
    """Unsloth-based training on our 9 category JSONLs.
    3 epochs, MoE tied LoRA, CCE, Mamba fast path. Tail-truncate to 8192 → 0% data loss."""
    import gc, json, math, sys, time, zipfile, random
    from pathlib import Path

    from unsloth import FastLanguageModel
    import kagglehub
    import torch
    from cut_cross_entropy import linear_cross_entropy
    from peft import LoraConfig
    from peft.tuners.lora import Linear as LoraLinear
    from transformers import AutoTokenizer

    # ── GPU + Mamba kernel sanity ────────────────────────────────────
    import causal_conv1d, mamba_ssm
    cc = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}  sm_{cc[0]*10+cc[1]}")
    print(f"torch={torch.__version__}  cuda={torch.version.cuda}")
    print(f"mamba_ssm={mamba_ssm.__version__}  causal_conv1d={causal_conv1d.__version__}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    from causal_conv1d import causal_conv1d_fn
    _x = torch.randn(1, 256, 32, device="cuda", dtype=torch.bfloat16)
    _w = torch.randn(256, 4, device="cuda", dtype=torch.bfloat16)
    causal_conv1d_fn(_x, _w, None, activation="silu")
    print("causal_conv1d CUDA kernel: OK")

    # ── Locate model + data ──────────────────────────────────────────
    MODEL_PATH = kagglehub.model_download(
        "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
    )

    data_dir = None
    for c in DATA_DIR_CANDIDATES:
        if c and os.path.isdir(c) and any(
            os.path.exists(os.path.join(c, f)) for f in CATEGORY_FILES
        ):
            data_dir = c
            break
    assert data_dir, f"no data dir found; searched: {DATA_DIR_CANDIDATES}"
    print(f"Data dir: {data_dir}")

    # ── Load 9 category JSONLs ───────────────────────────────────────
    raw = []
    for fname in CATEGORY_FILES:
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            print(f"  [skip] {fname}"); continue
        cat = fname.replace("train_cot_", "").replace(".jsonl", "")
        n = 0
        with open(fpath) as f:
            for line in f:
                if not line.strip(): continue
                rec = json.loads(line)
                rec.setdefault("category", cat)
                raw.append(rec); n += 1
        print(f"  {n:>5} from {fname}")
    print(f"Total raw records: {len(raw)}")

    # ── Tokenize: chat template + response-only loss mask + tail-truncate ──
    tok = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token

    examples = []
    truncated = 0; skipped = 0
    for rec in raw:
        msgs = [m for m in rec["messages"] if m["role"] != "system"]
        if not msgs or msgs[-1]["role"] != "assistant":
            skipped += 1; continue
        prompt_msgs = msgs[:-1]
        try:
            prompt_text = tok.apply_chat_template(
                prompt_msgs, tokenize=False, add_generation_prompt=True
            )
            full_text = tok.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False
            )
        except Exception:
            prompt_text = (
                f"<|im_start|>user\n{prompt_msgs[0]['content']}<|im_end|>\n"
                f"<|im_start|>assistant\n"
            )
            full_text = prompt_text + msgs[-1]["content"] + "<|im_end|>"

        prompt_ids = tok(prompt_text, add_special_tokens=False)["input_ids"]
        full_ids   = tok(full_text,   add_special_tokens=False)["input_ids"]
        if len(full_ids) <= len(prompt_ids):
            skipped += 1; continue

        # Tail-truncate: keep the last MAX_SEQ_LEN tokens — preserves the answer
        # which is at the END of the CoT. ZERO data loss across the corpus.
        if len(full_ids) > MAX_SEQ_LEN:
            cut = len(full_ids) - MAX_SEQ_LEN
            full_ids = full_ids[cut:]
            plen = max(0, len(prompt_ids) - cut)
            truncated += 1
        else:
            plen = len(prompt_ids)

        mask = [0] * plen + [1] * (len(full_ids) - plen)
        if not any(mask):
            skipped += 1; continue

        examples.append({
            "category": rec["category"],
            "tokens":   full_ids[:-1],
            "targets":  full_ids[1:],
            "weights":  [float(m) for m in mask[1:]],
        })

    total_tok = sum(len(e["tokens"]) for e in examples)
    total_lbl = sum(sum(e["weights"]) for e in examples)
    print(f"\nBuilt {len(examples)} examples (truncated={truncated}, skipped={skipped})")
    print(f"  total tokens   : {total_tok:,}")
    print(f"  loss-bearing   : {total_lbl:,.0f} ({100*total_lbl/total_tok:.1f}%)")

    # ── Load base model via Unsloth ──────────────────────────────────
    gc.collect(); torch.cuda.empty_cache()
    model, _ = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False, load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=True,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )

    # ── Wrap in LoRA ─────────────────────────────────────────────────
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK, target_modules=TARGET_MODULES,
        lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", use_gradient_checkpointing="unsloth", random_state=42,
    )
    FastLanguageModel.for_training(model)

    # ── Force Mamba CUDA fast path ───────────────────────────────────
    nemotron_mod = None
    for _name, _m in sys.modules.items():
        if "modeling_nemotron_h" in _name and hasattr(_m, "is_fast_path_available"):
            nemotron_mod = _m; break
    assert nemotron_mod is not None, "modeling_nemotron_h not loaded"
    nemotron_mod.is_fast_path_available = True
    print("Mamba fast path: ENABLED")

    # ── Add LoRA to lm_head (Unsloth drops it for MoE models) ────────
    _causal = model
    while hasattr(_causal, "model"):
        _causal = _causal.model
    if not isinstance(_causal.lm_head, LoraLinear):
        cfg = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
        model.base_model._create_and_replace(
            cfg, "default", target=_causal.lm_head, target_name="lm_head", parent=_causal,
        )
        print("Added LoRA to lm_head")

    # ── Cast LoRA → fp32; verify base bf16; verify MoE router fp32 ──
    for name, p in model.named_parameters():
        if ".lora_" in name:
            p.data = p.data.to(torch.float32)
    for name, p in model.named_parameters():
        if ".lora_" in name:
            assert p.dtype == torch.float32, f"{name} expected fp32"
            continue
        if ".mixer.gate." in name:
            assert p.dtype == torch.float32, f"router {name} expected fp32"
            continue
        assert p.dtype == torch.bfloat16, f"{name} expected bf16, got {p.dtype}"
    print("dtype check: LoRA fp32 / base bf16 / router fp32 ✓")

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Model: {trainable:,} trainable / {total:,} total")

    # ── CCE forward patch — no [B,T,131072] logits tensor ────────────
    _base = model
    while hasattr(_base, "model"):
        _base = _base.model

    def _cce_forward(input_ids=None, attention_mask=None, labels=None, **kw):
        out = _base.backbone(
            input_ids=input_ids, attention_mask=attention_mask,
            **{k: v for k, v in kw.items()
               if k in ("position_ids", "past_key_values", "use_cache")},
        )
        h = out[0]
        lh = _base.lm_head
        base_w = lh.base_layer.weight
        lA = lh.lora_A["default"].weight
        lB = lh.lora_B["default"].weight
        sc = lh.scaling["default"]
        lm_w = base_w + sc * lB @ lA
        if labels is not None:
            per_tok = linear_cross_entropy(h, lm_w, labels, reduction="none")
            loss = per_tok.mean()
        else:
            per_tok = None; loss = None
        model._cached_per_token_ce = per_tok
        return loss

    _base.forward = _cce_forward
    print("CCE forward patched — no logits materialization")

    # ── MoE tied LoRA (Tinker convention) ────────────────────────────
    moe_tied = []
    if MOE_TIE_WEIGHTS:
        w1_names = ("gate_up_proj", "up_proj", "gate_proj", ".w1.")
        w2_names = ("down_proj", ".w2.")
        for name, p in model.named_parameters():
            if not p.requires_grad or ".experts." not in name or ".lora_" not in name:
                continue
            is_w1 = any(s in name for s in w1_names)
            is_w2 = any(s in name for s in w2_names)
            is_A = ".lora_A." in name; is_B = ".lora_B." in name
            tie = (is_w1 and is_A) or (is_w2 and is_B)
            if tie and p.dim() >= 2 and p.shape[0] > 1:
                moe_tied.append(p)
        with torch.no_grad():
            for p in moe_tied:
                m = p.data.mean(dim=0, keepdim=True)
                p.data.copy_(m.expand_as(p.data))
        print(f"MoE tied params: {len(moe_tied)}")

    def _tie_grads():
        if not moe_tied: return
        with torch.no_grad():
            for p in moe_tied:
                if p.grad is None: continue
                g = p.grad.sum(dim=0, keepdim=True)
                p.grad.copy_(g.expand_as(p.grad))

    # ── Training loop: 3 epochs, shuffled per epoch ──────────────────
    gc.collect(); torch.cuda.empty_cache()
    device = next(model.parameters()).device

    steps_per_epoch = len(examples) // BATCH_SIZE
    max_steps = steps_per_epoch * NUM_EPOCHS
    print(f"Training: {NUM_EPOCHS} epochs × {steps_per_epoch} steps = {max_steps} total")
    print(f"  micro={MICRO_BATCH_SIZE}  batch={BATCH_SIZE}  lr={LEARNING_RATE}  max_seq={MAX_SEQ_LEN}")

    optimizer = None
    step = 0
    log_lines = []
    t_start = time.time()

    for epoch in range(NUM_EPOCHS):
        rng = random.Random(42 + epoch)
        order = list(range(len(examples)))
        rng.shuffle(order)
        print(f"\n=== Epoch {epoch+1}/{NUM_EPOCHS} (shuffled, seed={42+epoch}) ===")

        for batch_start in range(0, len(order), BATCH_SIZE):
            if step >= max_steps: break
            bi = order[batch_start:batch_start + BATCH_SIZE]
            if len(bi) < BATCH_SIZE: break
            batch = [examples[i] for i in bi]
            n_accum = math.ceil(len(batch) / MICRO_BATCH_SIZE)
            tot_loss_sum = 0.0; tot_w = 0.0

            for ms in range(0, len(batch), MICRO_BATCH_SIZE):
                mb = batch[ms:ms + MICRO_BATCH_SIZE]
                n = len(mb)
                ml = max(len(e["tokens"]) for e in mb)
                pi = torch.zeros(n, ml, dtype=torch.long, device=device)
                pt = torch.zeros(n, ml, dtype=torch.long, device=device)
                pw = torch.zeros(n, ml, dtype=torch.float32, device=device)
                am = torch.zeros(n, ml, dtype=torch.long, device=device)
                for i, e in enumerate(mb):
                    L = len(e["tokens"])
                    pi[i, :L] = torch.tensor(e["tokens"], dtype=torch.long)
                    pt[i, :L] = torch.tensor(e["targets"], dtype=torch.long)
                    pw[i, :L] = torch.tensor(e["weights"], dtype=torch.float32)
                    am[i, :L] = 1

                with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                    model(input_ids=pi, attention_mask=am, labels=pt, use_cache=False)
                    per_tok = model._cached_per_token_ce
                    weighted = per_tok * pw
                    ws = pw.sum(); ls = weighted.sum()
                    loss = ls / ws if ws > 0 else ls * 0.0
                (loss / n_accum).backward()
                tot_loss_sum += ls.item(); tot_w += ws.item()
                del loss, per_tok, weighted

            if optimizer is None:
                optimizer = torch.optim.AdamW(
                    [p for p in model.parameters() if p.requires_grad],
                    lr=LEARNING_RATE, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.0,
                )
            lr = LEARNING_RATE * (1 - step / max_steps)
            for pg in optimizer.param_groups: pg["lr"] = lr
            _tie_grads()
            gn = torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], max_norm=1.0
            )
            optimizer.step(); optimizer.zero_grad()
            mean_loss = tot_loss_sum / tot_w if tot_w > 0 else 0.0
            step += 1
            elapsed = time.time() - t_start
            eta = elapsed / step * (max_steps - step) if step > 0 else 0
            peak = torch.cuda.max_memory_allocated() / 1e9
            msg = (f"ep{epoch+1} step {step}/{max_steps} "
                   f"loss={mean_loss:.4f} grad_norm={gn:.3f} lr={lr:.2e} "
                   f"peak={peak:.1f}GB elapsed={elapsed/60:.1f}m eta={eta/60:.1f}m")
            print(msg, flush=True); log_lines.append(msg)

    print(f"\nTraining done. Total: {(time.time()-t_start)/3600:.2f} hrs")
    print(f"Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

    # ── Save adapter + rename lm_head keys for Nemotron-H eval ──────
    from safetensors.torch import load_file, save_file
    save_dir = "."
    for f in os.listdir(save_dir):
        if f.startswith("adapter"):
            os.remove(os.path.join(save_dir, f))
    model.save_pretrained(save_dir)
    st = os.path.join(save_dir, "adapter_model.safetensors")
    tensors = load_file(st)
    renamed = {
        k.replace("base_model.model.lm_head.", "base_model.model.backbone.lm_head."): v
        for k, v in tensors.items()
    }
    save_file(renamed, st)

    # cleanup unsloth compile cache
    import shutil as _sh
    if os.path.isdir("unsloth_compiled_cache"):
        _sh.rmtree("unsloth_compiled_cache")

    # zip submission
    adapter_files = [f for f in os.listdir(save_dir) if f.startswith("adapter")]
    with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        for f in adapter_files:
            zf.write(os.path.join(save_dir, f), f)
    for f in adapter_files:
        os.remove(os.path.join(save_dir, f))

    with open("training_log.txt", "w") as f:
        f.write("\n".join(log_lines))

    print("Wrote submission.zip + training_log.txt")


In [ ]:
# ── Cell 5: Run ─────────────────────────────────────────────────────
if IS_KAGGLE:
    run_training()
